# Importação das bibliotecas, definição do catálogo e importação da base

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, concat, lit
import pyspark.pandas as ps
import sys

## Definição do schema

In [0]:
%sql
SELECT current_user(), current_catalog(), current_schema();

USE CATALOG workspace;
USE SCHEMA ey_academy

## Bases a serem utilizadas (PremReg e SinReg)

In [0]:
df_prem = pd.read_csv("/Volumes/workspace/ey_academy/dados_analisados/PremReg_total.csv", sep=";")
df_sin = pd.read_csv("/Volumes/workspace/ey_academy/dados_analisados/SinReg_total.csv", sep=";")

### Na regulamentação da SUSEP, o termo “casco” no seguro de automóveis significa a cobertura para danos ao próprio veículo segurado — seja por colisão, incêndio, roubo, furto ou eventos da natureza.  Em outras palavras, é a proteção do bem em si, diferente das coberturas de responsabilidade civil (que protegem contra danos a terceiros).


### Significado das siglas de forma clara e direta
Essas siglas representam diferentes **coberturas de seguro de automóveis** regulamentadas pela SUSEP. Em resumo:  
- **APP**: protege os passageiros do veículo em caso de acidente.  
- **CASCO**: cobre danos ao próprio veículo segurado.  
- **RCDM**: cobre prejuízos materiais causados a terceiros.  
- **RCDP**: cobre danos pessoais e morais causados a terceiros.  
  
---  
  
### 📌 Explicação detalhada de cada categoria  
  
#### 🚑 **APP – Acidentes Pessoais de Passageiros**  
- Cobertura voltada para **pessoas transportadas no veículo segurado**.  
- Garante indenização em caso de **morte acidental** ou **invalidez permanente** dos passageiros em decorrência de acidente com o veículo.  
- É uma proteção adicional, diferente do seguro obrigatório DPVAT.  
  
---  
  
#### 🚗 **CASCO**  
- Refere-se ao **seguro do próprio veículo** do segurado.  
- Cobre **danos materiais** ao automóvel em situações como:  
  - Colisão.  
  - Incêndio.  
  - Roubo ou furto.  
  - Fenômenos da natureza (enchente, granizo, queda de árvore).  
- Pode ser contratado em diferentes modalidades de indenização:  
  - **Valor de Mercado Referenciado (VMR)**: baseado em tabela (ex.: FIPE).  
  - **Valor Determinado**: valor fixo acordado na apólice.  
  
---
  
#### 🏠 **RCDM – Responsabilidade Civil Facultativa de Veículos – Danos Materiais**
- Protege o segurado contra **prejuízos materiais causados a terceiros**.  
- Exemplos:  
  - Bater em outro carro e ter que pagar o conserto.  
  - Danificar um muro, poste ou outro bem material.  
- A seguradora indeniza o terceiro até o limite contratado.  

---

#### 👥 **RCDP – Responsabilidade Civil Facultativa de Veículos – Danos Pessoais e Morais**
- Cobre **danos corporais e morais** causados a terceiros em acidentes.  
- Exemplos:  
  - Atropelamento com despesas médicas e hospitalares.  
  - Indenização por invalidez ou morte de terceiros.  
  - Reclamações por danos morais decorrentes do acidente.  
- É uma proteção essencial para evitar que o segurado arque sozinho com custos elevados de indenizações.  

---

### ⚖️ Resumindo
- **APP** → protege os passageiros.  
- **CASCO** → protege o veículo do segurado.  
- **RCDM** → cobre danos materiais a terceiros.  
- **RCDP** → cobre danos pessoais e morais a terceiros.  

Essas coberturas podem ser contratadas isoladamente ou em conjunto, dependendo da necessidade do segurado e do tipo de apólice oferecida pela seguradora.  

---

[SUSEP – Seguro de Automóveis](https://www.gov.br/susep/pt-br/assuntos/meu-futuro-seguro/seguros-previdencia-e-capitalizacao/seguros/seguro-de-automoveis)  
[Condições Gerais – Seguro Automóvel (exemplo de seguradora)](https://allsegseguradora.com.br/wp-content/uploads/2025/06/CG-AUTO-CASCO-RCF-APP-v.fev25v2.pdf)  
[Circular SUSEP nº 639/2021](https://www.legisweb.com.br/noticia/?legislacao=418833)  


###Tabela PremReg
EXPOSICAO – Quantidade de veículos expostos (O conceito de exposição leva em conta o tempo em que cada apólice esteve vigente, dentro da janela de observação, que é o período semestral abrangido em cada atualização do Autoseg. Desta forma, o número de expostos, apurado para um período anual, representa o melhor estimador disponível para a quantidade de veículos segurados.)  
PREMIO – Soma dos valores de prêmios, ponderados pela exposição de cada apólice  
IS_MEDIA – Média das Importâncias Seguradas das apólices incluídas no grupamento definido pela chave escolhida, ponderada pela exposição de cada uma delas  
TIPO_PREM – Categoria APP - Acidentes Pessoais Passageiros, CASCO, RCDM - Responsabilidade Civil Facultativa de Veículos – Danos Materiais,
RCDP - Responsabilidade Civil Facultativa de Veículos – Danos Pessoais e Morais.

In [0]:
df_prem.head()
df_prem.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   estado_regiao  135 non-null    object
 1   tipo_premio    135 non-null    object
 2   exposicao      135 non-null    object
 3   premios        135 non-null    object
 4   is_media       135 non-null    object
dtypes: object(5)
memory usage: 5.4+ KB


In [0]:
df_prem.head()

,estado_regiao,tipo_premio,exposicao,premios,is_media
0,AC,APP,"10801,58889","623741,0727","84567,23089"
1,AC,CASCO,"14946,22992","26145505,01","116679,6815"
2,AC,RCDM,"16031,15046","4516861,413","180901,8069"
3,AC,RCDP,"15858,31484","1762863,793","180171,6139"
4,AC,TOTAL,"16414,94223","33047638,6","516737,4195"


### Tabela SinReg
TIPO_SIN – Variável que identifica a cobertura associada ao sinistro: APP, CASCO, RCDM e RCDP;  
NUNSINISTROS – Quantidade de sinistros;  
INDENIZACOES – Total de indenizações de sinistros.


In [0]:
df_sin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   estados_regiao  130 non-null    object
 1   tipo_sinistro   130 non-null    object
 2   num_sinistros   130 non-null    int64 
 3   indenizacoes    130 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 4.2+ KB


In [0]:
df_sin.head()

,estados_regiao,tipo_sinistro,num_sinistros,indenizacoes
0,AC,APP,1283,10109946
1,AC,CASCO,1855,12391540
2,AC,RCDM,334,2109096
3,AC,RCDP,1603,12206203
4,AL,TOTAL,1881,12476697


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, trim

spark = SparkSession.builder.appName("PremiosLimpeza").getOrCreate()

# Carregar com Spark, não Pandas
df_prem = spark.read.csv(
    "/Volumes/workspace/ey_academy/dados_analisados/PremReg_total.csv",
    sep=";",
    header=True,
    inferSchema=False
)

df_prem = df_prem.withColumn("premios_str", trim(col("premios")))
df_prem = df_prem.withColumn("premios_str", regexp_replace(col("premios_str"), r"[R\$\s\u00A0]", ""))
df_prem = df_prem.withColumn("premios_str", regexp_replace(col("premios_str"), r"\.", ""))
df_prem = df_prem.withColumn("premios_str", regexp_replace(col("premios_str"), r",", "."))
df_prem = df_prem.withColumn("premios", col("premios_str").cast("double")).drop("premios_str")

## Renomeando colunas "REGIAO" para "ESTADO" e criando coluna "REGIAO"

In [0]:
spark = SparkSession.builder.appName("EstadosRegioes").getOrCreate()

# Carregar tabelas
df_prem = spark.read.csv(
    "/Volumes/workspace/ey_academy/dados_analisados/PremReg_total.csv",
    sep=";",
    header=True,
    inferSchema=True
)

df_sin = spark.read.csv(
    "/Volumes/workspace/ey_academy/dados_analisados/SinReg_total.csv",
    sep=";",
    header=True,
    inferSchema=True
)

# Renomear coluna 'regiao' para 'estado'
df_prem = df_prem.withColumnRenamed("estado_regiao", "estado")
df_sin  = df_sin.withColumnRenamed("estados_regiao", "estado")

# Criar coluna 'regiao' com base no estado
df_prem = df_prem.withColumn(
    "regiao",
    when(col("estado").isin("SP", "RJ", "MG", "ES"), "Sudeste")
    .when(col("estado").isin("PR", "SC", "RS"), "Sul")
    .when(col("estado").isin("DF", "GO", "MT", "MS"), "Centro-Oeste")
    .when(col("estado").isin("BA", "PE", "CE", "MA", "PB", "RN", "AL", "SE", "PI"), "Nordeste")
    .when(col("estado").isin("AM", "PA", "RO", "RR", "AP", "AC", "TO"), "Norte")
    .otherwise("Desconhecido")
)

df_sin = df_sin.withColumn(
    "regiao",
    when(col("estado").isin("SP", "RJ", "MG", "ES"), "Sudeste")
    .when(col("estado").isin("PR", "SC", "RS"), "Sul")
    .when(col("estado").isin("DF", "GO", "MT", "MS"), "Centro-Oeste")
    .when(col("estado").isin("BA", "PE", "CE", "MA", "PB", "RN", "AL", "SE", "PI"), "Nordeste")
    .when(col("estado").isin("AM", "PA", "RO", "RR", "AP", "AC", "TO"), "Norte")
    .otherwise("Desconhecido")
)

# Exibir resultado
df_prem.select("estado", "regiao").display(10, truncate=False)
df_sin.select("estado", "regiao").display(10, truncate=False)

estado,regiao
AC,Norte
AC,Norte
AC,Norte
AC,Norte
AC,Norte
AL,Nordeste
AL,Nordeste
AL,Nordeste
AL,Nordeste
AL,Nordeste


estado,regiao
AC,Norte
AC,Norte
AC,Norte
AC,Norte
AL,Nordeste
AL,Nordeste
AL,Nordeste
AL,Nordeste
AL,Nordeste
AM,Norte


In [0]:
# Renomeia colunas para 'estado'
df_prem = df_prem.withColumnRenamed("estado_regiao", "estado")
df_sin  = df_sin.withColumnRenamed("estados_regiao", "estado")

# Cria coluna 'regiao' com base no estado
def add_regiao(df):
    return df.withColumn(
        "regiao",
        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("PR","SC","RS"), "Sul")
        .when(col("estado").isin("DF","GO","MT","MS"), "Centro-Oeste")
        .when(col("estado").isin("BA","PE","CE","MA","PB","RN","AL","SE","PI"), "Nordeste")
        .when(col("estado").isin("AM","PA","RO","RR","AP","AC","TO"), "Norte")
        .otherwise("Desconhecido")
    )

df_prem = add_regiao(df_prem)
df_sin  = add_regiao(df_sin)

# Limpa e converte prêmios
df_prem = (
    df_prem
    .withColumn("premios", trim(col("premios")))
    .withColumn("premios", regexp_replace(col("premios"), r"[R\$\s\u00A0]", ""))  # remove símbolos e espaços
    .withColumn("premios", regexp_replace(col("premios"), r"\.", ""))             # remove milhar
    .withColumn("premios", regexp_replace(col("premios"), r",", "."))             # vírgula -> ponto
    .withColumn("premios", col("premios").cast("double"))
)

# Converte indenizações
df_sin = df_sin.withColumn("indenizacoes", col("indenizacoes").cast("double"))

# Agregação por região
prem_por_regiao = df_prem.groupBy("regiao").agg(_sum("premios").alias("total_premios"))
sin_por_regiao  = df_sin.groupBy("regiao").agg(_sum("indenizacoes").alias("total_indenizacoes"))

# Join e cálculo com porcentagem

df_result = (
    prem_por_regiao.join(sin_por_regiao, on="regiao", how="inner")
    .withColumn("total_premios", round(col("total_premios"), 2))
    .withColumn("total_indenizacoes", round(col("total_indenizacoes"), 2))
    .withColumn("sinistralidade", round((col("total_indenizacoes") / col("total_premios")) * 100, 2))
    .withColumn("sinistralidade", concat(col("sinistralidade"), lit("%")))

)

# Exibir resultado ordenado
df_result.orderBy("regiao").display()

regiao,total_premios,total_indenizacoes,sinistralidade
Centro-Oeste,5.44296743944E9,4.206239876E9,77.28%
Nordeste,7.54874565642E9,5.923609461E9,78.47%
Norte,1.271486541E9,9.49332055E8,74.66%
Sudeste,3.576184192255E10,2.446985136E10,68.42%
Sul,1.391855051114E10,9.816522034E9,70.53%


## Sinistralidade por região  
  
## Sinistralidade = Indenizações / Prêmios  
- Essa tabela traz uma visão consolidada da sinistralidade por região, ou seja, mostra como os prêmios arrecadados (receita das seguradoras) se comparam às indenizações pagas (despesas com sinistros).

📊 O que ela mostra
- Prêmios (total_premios): quanto foi arrecadado em cada região.
- Indenizações (total_indenizacoes): quanto foi pago em sinistros.
- Sinistralidade: razão entre indenizações e prêmios, indicando a sustentabilidade da operação.

🔎 Principais insights
- Nordeste e Centro-Oeste
- Sinistralidade mais alta (0.78 e 0.77).
- Isso significa que 77–78% dos prêmios arrecadados viraram indenizações.
- Margem menor para a seguradora, maior risco.
- Norte
- Sinistralidade de 0.75, também elevada.
- Apesar de valores absolutos menores, proporcionalmente consome bastante da receita.
- Sul e Sudeste
- Sinistralidade mais baixa (0.71 e 0.68).
- Regiões mais rentáveis, especialmente o Sudeste, que concentra o maior volume de prêmios e mantém índice relativamente controlado.
- Sudeste
- Destaque absoluto em volume: arrecadou R$ 35,7 bilhões e pagou R$ 24,4 bilhões.
- Mesmo com maior exposição, mantém a menor sinistralidade (0.68), indicando carteira mais equilibrada.



### Sinistralidade por REGIÃO e CATEGORIA

####
O que chama atenção é que em algumas categorias (ex.: CASCO no Norte e Nordeste) a sinistralidade ultrapassa 100%, ou seja, as seguradoras pagaram mais em indenizações do que arrecadaram em prêmios → operação deficitária.
• 	Já em APP e RCDM, os índices são mais baixos, indicando maior equilíbrio.

In [0]:
from pyspark.sql import SparkSession
#from pyspark.sql.functions

# Renomear colunas para 'estado'
df_prem = df_prem.withColumnRenamed("estado_regiao", "estado")
df_sin  = df_sin.withColumnRenamed("estados_regiao", "estado")

# Criar coluna 'regiao' com base no estado
def add_regiao(df):
    return df.withColumn(
        "regiao",
        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("PR","SC","RS"), "Sul")
        .when(col("estado").isin("DF","GO","MT","MS"), "Centro-Oeste")
        .when(col("estado").isin("BA","PE","CE","MA","PB","RN","AL","SE","PI"), "Nordeste")
        .when(col("estado").isin("AM","PA","RO","RR","AP","AC","TO"), "Norte")
        .otherwise("Desconhecido")
    )

df_prem = add_regiao(df_prem)
df_sin  = add_regiao(df_sin)

# Limpar e converter prêmios
df_prem = (
    df_prem
    .withColumn("premios", trim(col("premios")))
    .withColumn("premios", regexp_replace(col("premios"), r"[R\\$\\s\\u00A0]", ""))  # remove símbolos
    .withColumn("premios", regexp_replace(col("premios"), r"\\.", ""))               # remove milhar
    .withColumn("premios", regexp_replace(col("premios"), r",", "."))                # vírgula -> ponto
    .withColumn("premios", col("premios").cast("double"))
)

# Converter indenizações
df_sin = df_sin.withColumn("indenizacoes", col("indenizacoes").cast("double"))

# Agregar por categoria e região
prem_por_cat_reg = df_prem.groupBy("regiao", "tipo_premio").agg(_sum("premios").alias("total_premios"))
sin_por_cat_reg  = df_sin.groupBy("regiao", "tipo_sinistro").agg(_sum("indenizacoes").alias("total_indenizacoes"))

# Usar aliases para evitar ambiguidade
prem_por_cat_reg = prem_por_cat_reg.alias("prem")
sin_por_cat_reg  = sin_por_cat_reg.alias("sin")

# Join por região e categoria
df_join = prem_por_cat_reg.join(
    sin_por_cat_reg,
    (col("prem.regiao") == col("sin.regiao")) &
    (col("prem.tipo_premio") == col("sin.tipo_sinistro")),
    how="inner"
)

# Calcular sinistralidade como porcentagem
df_result = (
    df_join
    .withColumn("total_premios", round(col("total_premios"), 2))
    .withColumn("total_indenizacoes", round(col("total_indenizacoes"), 2))
    .withColumn("sinistralidade", round((col("total_indenizacoes") / col("total_premios")) * 100, 2))
    .withColumn("sinistralidade", concat(col("sinistralidade"), lit("%")))
    .select(col("prem.regiao").alias("regiao"),
            col("prem.tipo_premio").alias("tipo_premio"),
            "total_premios", "total_indenizacoes", "sinistralidade")
)

# Exibir resultado
df_result.orderBy("regiao", "tipo_premio").display(truncate=False)

regiao,tipo_premio,total_premios,total_indenizacoes,sinistralidade
Centro-Oeste,APP,5.984447672E7,2400818.0,4.01%
Centro-Oeste,CASCO,2.1562288714E9,1.615286886E9,74.91%
Centro-Oeste,RCDM,4.3067306479E8,4.67951949E8,108.66%
Centro-Oeste,RCDP,1.4935437929E8,1.7480285E7,11.7%
Centro-Oeste,TOTAL,3.4307586493E9,2.103119938E9,61.3%
Nordeste,APP,2.132044983E7,1656961.0,7.77%
Nordeste,CASCO,3.0831013885E9,2.422141681E9,78.56%
Nordeste,RCDM,5.584764216E8,5.1861627E8,92.86%
Nordeste,RCDP,1.6201679483E8,1.314147E7,8.11%
Nordeste,TOTAL,3.8956198864E9,2.968053079E9,76.19%


In [0]:
# Renomear colunas para 'estado'
df_prem = df_prem.withColumnRenamed("estado_regiao", "estado")
df_sin  = df_sin.withColumnRenamed("estados_regiao", "estado")

# Criar coluna 'regiao' com base no estado
def add_regiao(df):
    return df.withColumn(
        "regiao",
        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("PR","SC","RS"), "Sul")
        .when(col("estado").isin("DF","GO","MT","MS"), "Centro-Oeste")
        .when(col("estado").isin("BA","PE","CE","MA","PB","RN","AL","SE","PI"), "Nordeste")
        .when(col("estado").isin("AM","PA","RO","RR","AP","AC","TO"), "Norte")
        .otherwise("Desconhecido")

# Garantir que colunas 'regiao' e 'estado' já existem (como informado)
# Se não existirem, você pode adicionar aqui a lógica para criar

# Limpeza e conversão de prêmios
df_prem["premios"] = (
    df_prem["premios"]
    .str.replace(r"[R\$\s\u00A0]", "", regex=True)
    .str.replace(r"\.", "", regex=True)
    .str.replace(",", ".", regex=True)
    .astype(float)
)

# Converte indenizações
df_sin["indenizacoes"] = df_sin["indenizacoes"].astype(float)

# Agregar por região e categoria
prem_cat = df_prem.groupby(["regiao", "tipo_premio"]).agg({"premios": "sum"}).reset_index()
sin_cat = df_sin.groupby(["regiao", "tipo_sinistro"]).agg({"indenizacoes": "sum"}).reset_index()

# Unir categorias (assumindo que tipo_premio e tipo_sinistro são equivalentes)
df_cat = prem_cat.merge(sin_cat, left_on=["regiao", "tipo_premio"], right_on=["regiao", "tipo_sinistro"], how="inner")

# Calcular sinistralidade por categoria
df_cat["sinistralidade"] = df_cat["indenizacoes"] / df_cat["premios"]

# Pivotar para comparação
pivot = df_cat.pivot(index="tipo_premio", columns="regiao", values="sinistralidade")

# Calcular proximidade com Sul
sul_values = pivot["Sul"]
distancias = pivot.apply(lambda col: abs(col - sul_values).mean(), axis=0)

# Região mais próxima da Sul
regiao_proxima = distancias.drop("Sul").idxmin()
print("Região mais próxima da Sul:", regiao_proxima)
print("Distâncias por região:\n", distancias)

  File <command-8780083086433708>, line 7
    return df.withColumn(
                        ^
SyntaxError: '(' was never closed


###
- Regiões com sinistralidade alta (Nordeste, Centro-Oeste, Norte):
- Podem exigir revisão de preços, políticas de subscrição ou maior foco em prevenção de sinistros.
- Regiões com sinistralidade baixa (Sudeste, Sul):
- Mostram maior rentabilidade e sustentabilidade da operação.
- Visão geral:
- Todas as regiões têm sinistralidade relativamente alta (acima de 0.68), o que indica que o mercado de automóveis é desafiador e exige gestão cuidadosa de risco.

A tabela mostra quais regiões são mais rentáveis e quais apresentam maior risco para a seguradora, permitindo direcionar estratégias de preço, subscrição e prevenção.


A sinistralidade tem impacto no valor do prêmio devido quantidade de indenizações pago em relação 